In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr,hash,lit

In [2]:
spark = SparkSession.builder.appName('practise').getOrCreate()

25/04/22 10:51:41 WARN Utils: Your hostname, nischit-baral resolves to a loopback address: 127.0.1.1; using 10.10.42.113 instead (on interface enp2s0)
25/04/22 10:51:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/22 10:51:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

### Load the given JSON into pyspark data-frame



In [4]:
df_pyspark = spark.read.option('header','true').json('MOCK_DATA.json')

In [5]:
df_pyspark.show()

+--------------------+-------------+-----------------+-------------+--------------------+---------------+---------------+---------------+-----------------------+------------+
|     _corrupt_record|billing_class|billing_code_type|billing_codee|         description|expiration_date|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+--------------------+-------------+-----------------+-------------+--------------------+---------------+---------------+---------------+-----------------------+------------+
|[{"billing_class"...|         NULL|             NULL|         NULL|                NULL|           NULL|           NULL|           NULL|                   NULL|        NULL|
|                NULL|          NVA|             NULL|   54868-6381|Occup of pk-up/va...|     2023/10/27|          $1.89|             CO|             49288-0290|          ID|
|                NULL|          JEQ|             SNJK|    55315-329|Posterior subluxa...|     2023/03/22|          $6.07|    

### Remove name, description and expiration date


In [6]:
df = df_pyspark.drop('name','description','expiration_date')

In [7]:
df.show()

+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|     _corrupt_record|billing_class|billing_code_type|billing_codee|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|[{"billing_class"...|         NULL|             NULL|         NULL|           NULL|           NULL|                   NULL|        NULL|
|                NULL|          NVA|             NULL|   54868-6381|          $1.89|             CO|             49288-0290|          ID|
|                NULL|          JEQ|             SNJK|    55315-329|          $6.07|             BR|              10544-105|          CN|
|                NULL|          SSO|             SNLO|    13537-015|          $4.78|             BR|              53489-536|          CN|
|                NULL|          CK

In [8]:

# df_pyspark.na.drop().show()

### Remove hyphen from billing code, negotiation arrangements 
### Remove $ from negotiated rate



In [9]:
df1 = df.withColumn('billing_codee', expr("replace(billing_codee,'-','')"))\
        .withColumn('negotiation_arrangement', expr("replace(negotiation_arrangement,'-','')"))\
        .withColumn('negotiated_rate', expr("replace(negotiated_rate,'$','')"))


In [10]:
df1.show()

+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|     _corrupt_record|billing_class|billing_code_type|billing_codee|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|[{"billing_class"...|         NULL|             NULL|         NULL|           NULL|           NULL|                   NULL|        NULL|
|                NULL|          NVA|             NULL|    548686381|           1.89|             CO|              492880290|          ID|
|                NULL|          JEQ|             SNJK|     55315329|           6.07|             BR|               10544105|          CN|
|                NULL|          SSO|             SNLO|     13537015|           4.78|             BR|               53489536|          CN|
|                NULL|          CK

### Use hashing for service code column


In [11]:
df2 = df1.withColumn('service_code',hash("service_code"))

In [12]:
df2.show()

+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|     _corrupt_record|billing_class|billing_code_type|billing_codee|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+--------------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|[{"billing_class"...|         NULL|             NULL|         NULL|           NULL|           NULL|                   NULL|          42|
|                NULL|          NVA|             NULL|    548686381|           1.89|             CO|              492880290|  -214147340|
|                NULL|          JEQ|             SNJK|     55315329|           6.07|             BR|               10544105|   446854828|
|                NULL|          SSO|             SNLO|     13537015|           4.78|             BR|               53489536|   446854828|
|                NULL|          CK

### Remove the row if 'billing_code' is null


In [13]:
df3 = df2.dropna(subset=["billing_codee"])

In [14]:
df3.show()

+---------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|_corrupt_record|billing_class|billing_code_type|billing_codee|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+---------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|           NULL|          NVA|             NULL|    548686381|           1.89|             CO|              492880290|  -214147340|
|           NULL|          JEQ|             SNJK|     55315329|           6.07|             BR|               10544105|   446854828|
|           NULL|          SSO|             SNLO|     13537015|           4.78|             BR|               53489536|   446854828|
|           NULL|          CKR|             NULL|     42508138|           4.35|             US|               59572425|   519803259|
|           NULL|          CXH|             CYHC|     52533120|      

### Replace all the null values in billing_class to ‘I’


In [15]:
# df4 = df3.withColumn('billing_class', when(col("billing_class").isNull(),I).otherwise(col")
df4 = df3.fillna({"billing_class":'I'})
df4.show()


+---------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|_corrupt_record|billing_class|billing_code_type|billing_codee|negotiated_rate|negotiated_type|negotiation_arrangement|service_code|
+---------------+-------------+-----------------+-------------+---------------+---------------+-----------------------+------------+
|           NULL|          NVA|             NULL|    548686381|           1.89|             CO|              492880290|  -214147340|
|           NULL|          JEQ|             SNJK|     55315329|           6.07|             BR|               10544105|   446854828|
|           NULL|          SSO|             SNLO|     13537015|           4.78|             BR|               53489536|   446854828|
|           NULL|          CKR|             NULL|     42508138|           4.35|             US|               59572425|   519803259|
|           NULL|          CXH|             CYHC|     52533120|      

### Rename all the column as given. billing_class>>bCls  , billing_code>>bC, billing_code_type>> bCT , negotiated_rate>> negR, negotiated_type>> negT, negotiation_arrangements>> negA, service_code>> poSH


In [16]:
rename_col = df4.withColumnRenamed('billing_class','bCIs')\
                .withColumnRenamed('billing_codee','bC')\
                .withColumnRenamed('billing_code_type','bCT')\
                .withColumnRenamed('negotiated_rate','negR')\
                .withColumnRenamed('negotiated_type','negT')\
                .withColumnRenamed('negotiation_arrangement','negA')\
                .withColumnRenamed('service_code','poSH')
rename_col.show()


+---------------+----+----+---------+----+----+---------+-----------+
|_corrupt_record|bCIs| bCT|       bC|negR|negT|     negA|       poSH|
+---------------+----+----+---------+----+----+---------+-----------+
|           NULL| NVA|NULL|548686381|1.89|  CO|492880290| -214147340|
|           NULL| JEQ|SNJK| 55315329|6.07|  BR| 10544105|  446854828|
|           NULL| SSO|SNLO| 13537015|4.78|  BR| 53489536|  446854828|
|           NULL| CKR|NULL| 42508138|4.35|  US| 59572425|  519803259|
|           NULL| CXH|CYHC| 52533120|1.35|  CA| 41250871| -214147340|
|           NULL| KTT|EFKT| 36800952|7.52|  FI| 59886410|  446854828|
|           NULL|   I|NULL| 11344999|2.94|  JP| 76509151|  446854828|
|           NULL| DAL|NULL|369871207|4.07|NULL| 11410020| 1062621904|
|           NULL| GDA|NULL| 76138106|9.05|  CF| 65954538| 1817792724|
|           NULL| AEX|KAEX| 53329809|3.61|  US| 50991216|  175481947|
|           NULL|   I|SPHI| 04960760|4.79|  PE| 49884641|-1524333386|
|           NULL| DU

### Add a whole new column named billing_code_modifier and rename it to 'mdH'


In [17]:
new_col = rename_col.withColumn('billing_code_modifier',lit(1426636))

In [18]:
new_col.show()

+---------------+----+----+---------+----+----+---------+-----------+---------------------+
|_corrupt_record|bCIs| bCT|       bC|negR|negT|     negA|       poSH|billing_code_modifier|
+---------------+----+----+---------+----+----+---------+-----------+---------------------+
|           NULL| NVA|NULL|548686381|1.89|  CO|492880290| -214147340|              1426636|
|           NULL| JEQ|SNJK| 55315329|6.07|  BR| 10544105|  446854828|              1426636|
|           NULL| SSO|SNLO| 13537015|4.78|  BR| 53489536|  446854828|              1426636|
|           NULL| CKR|NULL| 42508138|4.35|  US| 59572425|  519803259|              1426636|
|           NULL| CXH|CYHC| 52533120|1.35|  CA| 41250871| -214147340|              1426636|
|           NULL| KTT|EFKT| 36800952|7.52|  FI| 59886410|  446854828|              1426636|
|           NULL|   I|NULL| 11344999|2.94|  JP| 76509151|  446854828|              1426636|
|           NULL| DAL|NULL|369871207|4.07|NULL| 11410020| 1062621904|           

### Add a whole new column named billing_code_modifier and rename it to 'mdH'


In [19]:
renamming_df = new_col.withColumnRenamed('billing_code_modifier','mdH')

In [20]:
renamming_df.show()

+---------------+----+----+---------+----+----+---------+-----------+-------+
|_corrupt_record|bCIs| bCT|       bC|negR|negT|     negA|       poSH|    mdH|
+---------------+----+----+---------+----+----+---------+-----------+-------+
|           NULL| NVA|NULL|548686381|1.89|  CO|492880290| -214147340|1426636|
|           NULL| JEQ|SNJK| 55315329|6.07|  BR| 10544105|  446854828|1426636|
|           NULL| SSO|SNLO| 13537015|4.78|  BR| 53489536|  446854828|1426636|
|           NULL| CKR|NULL| 42508138|4.35|  US| 59572425|  519803259|1426636|
|           NULL| CXH|CYHC| 52533120|1.35|  CA| 41250871| -214147340|1426636|
|           NULL| KTT|EFKT| 36800952|7.52|  FI| 59886410|  446854828|1426636|
|           NULL|   I|NULL| 11344999|2.94|  JP| 76509151|  446854828|1426636|
|           NULL| DAL|NULL|369871207|4.07|NULL| 11410020| 1062621904|1426636|
|           NULL| GDA|NULL| 76138106|9.05|  CF| 65954538| 1817792724|1426636|
|           NULL| AEX|KAEX| 53329809|3.61|  US| 50991216|  17548

### Also write that dataframe into parquet using coalesce (1).


In [22]:
renamming_df.coalesce (1).write.parquet('new_data1')